In [ ]:
import asyncio
from telethon import TelegramClient
from telethon.errors import FloodWaitError
from key_telegram import username, api_id, api_hash
import random
import uuid
import pandas as pd
from datetime import datetime, timezone
import difflib
import re

client = TelegramClient(username, api_id, api_hash)

msg_counter = 0
msg_storage = {}
reserve = {}


def clean_text(text: str) -> str:
    # Видалення HTML-тегів
    text = re.sub(r"<[^>]+>", "", text)
    # Видалення посилань разом із дужками, якщо URL стоїть у дужках
    text = re.sub(r"\(\s*https?://[^\s)]+\s*\)", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    # Видалення @згадок
    text = re.sub(r"@\w+", "", text)
    # Видалення хештегів
    text = re.sub(r"#\w+", "", text)
    # Видалення конструкцій типу (!), (!!!), (?), (??)
    text = re.sub(r"\(\s*[!?]+\s*\)", "", text)
    # Видалення порожніх дужок
    text = re.sub(r"\(\s*\)", "", text)
    # Видалення невидимих службових символів
    text = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", text)
    # Видалення декоративних розділювачів
    text = re.sub(r"[━─═_~*]{2,}", "", text)
    # Видалення повторюваних декоративних знаків
    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)
    # Видалення зайвих пробілів
    text = re.sub(r"[ \t]+", " ", text)
    # Видалення пробілів перед розділовими знаками
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    # Прибирання пробілів на початку та в кінці рядків
    text = re.sub(r" *\n *", "\n", text)
    # Не більше двох переносів рядка поспіль
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def is_too_similar(new_text: str,
                   existing_texts: list,
                   threshold: float = 0.8) -> bool:
    for stored_text in existing_texts:
        similarity = difflib.SequenceMatcher(None, new_text, stored_text).ratio()
        if similarity >= threshold:
            return True
    return False


async def get_channel_messages(channel_name: str,
                               start_date: datetime,
                               end_date: datetime,
                               limit: int = 500,
                               min_words: int = 6,
                               max_words:int = 80):
    global msg_counter
    global msg_storage
    global reserve

    try:
        async for message in client.iter_messages(channel_name,
                                                  limit=limit,
                                                  offset_date=end_date):
            if message.date < start_date:
                break

            if not message.text:
                continue

            if message.fwd_from:
                continue

            cleaned_text = clean_text(message.text)

            if not cleaned_text:
                continue

            words = cleaned_text.split()
            if len(words) < min_words or len(words) > max_words:
                continue

            all_saved_texts = list(msg_storage.values()) + list(reserve.values())
            if is_too_similar(cleaned_text, all_saved_texts, threshold=0.8):
                continue

            msg_id = str(uuid.uuid4())
            if msg_counter < 2000:
                msg_storage[msg_id] = cleaned_text
                msg_counter += 1
            else:
                reserve[msg_id] = cleaned_text

    except FloodWaitError as e:
        await asyncio.sleep(e.seconds)


def save_to_csv(data: dict, file_name: str):
    if not data:
        return
    df = pd.DataFrame(data.items(), columns=["id", "text"])
    filename = f"{file_name}.csv"

    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"saved {len(df)} messages to: \"{filename}\"")


def save_to_excel(data: dict, file_name: str):
    if not data:
        return

    df = pd.DataFrame(data.items(), columns=["id", "text"])
    filename = f"{file_name}.xlsx"

    df.to_excel(filename, index=False, engine="openpyxl")
    print(f"saved {len(df)} messages to: \"{filename}\"")


async def main():
    global msg_storage
    global reserve

    channel_names = ["truexanewsua", "kievinfo_kyiv", "times_ukraina",
                      "voynareal", "oleksiihoncharenko", "reiters", "ssternenko"]

    start_date = datetime(2025, 9, 15, tzinfo=timezone.utc)
    end_date = datetime(2026, 9, 15, tzinfo=timezone.utc)
    limit = 800
    min_words = 10
    max_words = 80

    async with client:
        for channel_name in channel_names:

            await get_channel_messages(channel_name=channel_name,
                                                       start_date=start_date,
                                                       end_date=end_date,
                                                       limit=limit,
                                                       min_words=min_words,
                                                       max_words=max_words)
            print(f"finished processing {channel_name}")
            delay = random.uniform(3.0, 6.0)
            await asyncio.sleep(delay)

        save_to_csv(msg_storage, file_name="manipulation_dataset")
        save_to_excel(msg_storage, file_name="manipulation_dataset")

        save_to_csv(reserve, file_name="reserve")
        save_to_excel(reserve, file_name="reserve")


if __name__ == "__main__":
    asyncio.run(main())
